Agent LLM

1. Z wykorzystaniem biblioteki Open AI Agents SDK (https://openai.github.io/openai-agentspython/) stwórz agenta, który potęguje liczby. Wykorzystaj model „llama-3.1-8b-instant”

In [ ]:
from agents import Agent, Runner, function_tool, set_default_openai_api, set_default_openai_client, set_tracing_disabled
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)
set_default_openai_client(client=client, use_for_tracing=False)
set_default_openai_api("chat_completions")
set_tracing_disabled(disabled=True)

@function_tool
def power(x: float, y: float):
    return x ** y


agent = Agent(
    name="Assistant",
    instructions="",
    model="llama-3.1-8b-instant",
    tools=[power]
)

result = await Runner.run(agent, "2^2")
print(result.final_output)

If you meant the mathematical operation, 2^2 is equal to 4.


2. Zweryfikuj, czy dla kolejnych takich samych zapytań wynik zwracany jest w takiej samej formie. Jeżeli nie, to spraw, aby agent zwracał wynik potęgowania zawsze w takiej samej formie.

In [5]:
from time import sleep
agent.instructions = ""
result = await Runner.run(agent, "2^12")
print(result.final_output)
print("----------------")
sleep(2)
result = await Runner.run(agent, "5^2.1")
print(result.final_output)
print("----------------")
sleep(2)
result = await Runner.run(agent, "3^0.122")
print(result.final_output)
print("----------------")
sleep(2)



print("----------------------")
print("new instruction")
print("----------------------")

agent = Agent(
    name="Assistant",
    instructions="""
    Use power tool on given numbers.
    Return the answer in the format: Result: <result>
    """,
    model="llama-3.1-8b-instant",
    tools=[power]
)
sleep(2)
result = await Runner.run(agent, "2^12")
print(result.final_output)
print("----------------")
sleep(2)
result = await Runner.run(agent, "5^2.1")
print(result.final_output)
print("----------------")
sleep(2)
result = await Runner.run(agent, "3^0.122")
print(result.final_output)
print("----------------")
sleep(2)


However, if we want the actual value, which is an exponential of the given numbers, the final result would be 4096.
----------------
The result of 5^2.1 is approximately 29.36547357720048.
----------------
So the result of 3^0.122 is approximately 1.143427921448312.
----------------
----------------------
new instruction
----------------------
Result: 4096
----------------
Result: 29.36547357720048
----------------
Result: 1.143427921448312
----------------


3. Zweryfikuj, czy agent obsługuje następujące zapisy liczbowe: 0.123, 0,123, 1.23e-1. Jeżeli nie, to spraw, aby je obsługiwał.

In [7]:
agent = Agent(
    name="Assistant",
    instructions="""
    Use power tool on given numbers.
    Return the answer in the format: Result: <result>
    """,
    model="llama-3.1-8b-instant",
    tools=[power]
)
try:
    result = await Runner.run(agent, "2^0.123")
    print(result.final_output)
except:
    print("wrong response")
print("----------------")
try:
    result = await Runner.run(agent, "3^0,123")
    print(result.final_output)
except:
    print("wrong response")
print("----------------")
try:
    result = await Runner.run(agent, "4^0.123")
    print(result.final_output)
except:
    print("wrong response")
print("----------------")
try:
    result = await Runner.run(agent, "(2e-1)^(1.23e2)")
    print(result.final_output)
except:
    print("wrong response")
print("----------------")

agent.instructions = """
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125
    Return the answer in the format: Result: <result>
"""

print("---------------------------------")
print("new instruction")
print("---------------------------------")
result = await Runner.run(agent, "2^0.123")
print(result.final_output)
print("----------------")
result = await Runner.run(agent, "3^0,123")
print(result.final_output)
print("----------------")
result = await Runner.run(agent, "4^0.123")
print(result.final_output)
print("----------------")
result = await Runner.run(agent, "(2e-1)^(1.23e2)")
print(result.final_output)
print("----------------")


Result: 1.0889970153361064
----------------
Result: 1.1446847956963533
----------------
Result: 1.1859144994109478
----------------
wrong response
----------------
---------------------------------
new instruction
---------------------------------
Result: 1.0889970153361064
----------------
Result: 1.1446847956963533
----------------
Result: 1.1859144994109478
----------------
Result: 1.0633823966279399e-86
----------------


4. Sprawdź, czy agent odpowie poprawnie, gdy zapiszemy potęgę jako ułamek np. 4^(1/2). Jeżeli nie, to popraw agenta

In [8]:
agent = Agent(
    name="Assistant",
    instructions="""
    Use power tool on given numbers.
    Return the answer in the format: Result: <result>
    """,
    model="llama-3.1-8b-instant",
    tools=[power]
)
try:
    result = await Runner.run(agent, "4^(1/2)")
    print(result.final_output)
except:
    print("wrong response")
print("----------------")

agent.instructions = """
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125
    Return the answer in the format: Result: <result>
"""
result = await Runner.run(agent, "4^(1/2)")
print(result.final_output)

Result: 2.0
----------------
Result: 2.0


5. Sprawdź, czy agent odpowie poprawnie dla 4^(1/0), gdzie w potędze występuje jest dzielenie przez 0. Jeżeli nie, to popraw agenta.

In [21]:
try:
    result = await Runner.run(agent, "4^(1/0)")
    print(result.final_output)
except:
    print("wrong response")
print("----------------")

print("new instruction")
print("----------------")

sleep(2)
agent.instructions = """
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    Return the answer in the format: Result: <result>
"""
result = await Runner.run(agent, "4^(1/0)")
print(result.final_output)

Given numbers are incorrect.</function>
----------------
new instruction
----------------
Given numbers are incorrect.


6. Sprawdź, czy agent szanuje kolejność działań? Np. 16^1/2 (najpierw potęgowanie). Jeżeli nie, to popraw agenta

In [28]:
agent.instructions = """
    Use power tool on given numbers.
    IMPORTANT RULES for number formatting:
    1. The tool ONLY accepts decimal numbers (floats). It does NOT accept fractions like '1/3'.
    2. If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    3. Parse formats like 1,23 to 1.23.
    4. Scientific notation like 1.23e-1 is allowed.

    If the second given number is mathematically impossible (like division by zero), say that given numbers are incorrect.
    Return the answer in the format: Result: <result>
"""

result = await Runner.run(agent, "8^1/3")
print(result.final_output)

Result: 1.999861375368307


7. Spraw, aby wynik zwracany przez agenta zawsze zawierał 6 miejsc po przecinku.

In [31]:
agent.instructions = """
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    Return the answer in the format: Result: <result>. Result must have 6 decimal places.
"""

result = await Runner.run(agent, "8^1/2")
print(result.final_output)
sleep(2)
result = await Runner.run(agent, "5^1/2")
print(result.final_output)


Result: 2.828427
Result: 2.236068


8. Zweryfikuj, czy agent obsługuje kilka działań w jednym zapytaniu. Jeżeli nie, to spraw, aby
obsługiwał. 

In [34]:
agent.instructions = """
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    Return the answer in the format: Result: <result>. Result must have 6 decimal places.
"""
try:
    result = await Runner.run(agent, "4^(16^(1/2))")
    print(result.final_output)
except:
    print("wrong response")

print("------------------")
print("new instruction")
print("------------------")
sleep(2)
agent.instructions = """
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    If given nested expressions use power tool on them accordingly.
    Example process for 3^(8^(1/3)):
    Step 1: Calculate 8^(1/3). Call power(8, 0.333333). System returns 2.
    Step 2: Calculate 3^2 (using the result from Step 1). Call power(3, 2). System returns 9.
    Return the answer in the format: Result: <result>. Result has to have 6 decimal places.
"""
result = await Runner.run(agent, "4^(16^(1/2))")
print(result.final_output)

Result: 262144.000000
------------------
new instruction
------------------
Result: 256.000000


9. Sprawdź co się stanie, jeżeli zapytanie użytkownika będzie pustym napisem

In [43]:
result = await Runner.run(agent, "")
print(result.final_output)

But we must calculate it as 81.000000


10. Sprawdź co się stanie, jeżeli zapytanie użytkownika nie dotyczy potęgowania

In [42]:
result = await Runner.run(agent, "What happened on the 4th of June 1989?")
print(result.final_output)

brave_search{"query": "4th june 1989"}


11. Dodaj do agenta narzędzie pozwalające mu sprawdzenie pogody w zadanej miejscowości. Sprawdź za pomocą agenta jaka jest obecna pogoda w Gliwicach.

In [ ]:
WEATHER_API_KEY="xxxxxxx"
WEATHER_URL="https://api.weatherapi.com/"

In [50]:
import requests
@function_tool
def current_weather(place):
    current = requests.get(WEATHER_URL+"/v1/current.json", params={
        "key": WEATHER_API_KEY,
        "q": place,
    }).json()["current"]
    return current

agent.tools = [power, current_weather]

agent.instructions = """
You can also tell the weather
"""

result = await Runner.run(agent, "What's the weather today in Gliwice?")
print(result.final_output)

That's the current weather in Gliwice. It's overcast with a temperature of 11.2 degrees Celsius and a humidity of 82%. The wind speed is 18 km/h from the southwest at 1017 mbar pressure and 30.03 inches.


12. Sprawdź, czy agent potrafi jednocześnie potęgować i podać informacje o pogodzie?


In [51]:
agent.instructions = """
    You are a multi-tasking assistant. Your goal is to identify ALL distinct requests in the user's input and execute them.

    ### Task 1: Mathematical Calculations
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    If given nested expressions use power tool on them accordingly.
    Example process for 3^(8^(1/3)):
    Step 1: Calculate 8^(1/3). Call power(8, 0.333333). System returns 2.
    Step 2: Calculate 3^2 (using the result from Step 1). Call power(3, 2). System returns 9.
    **Math Output**: Return the answer in the format: Result: <result>. Result has to have 6 decimal places.

    ### Task 2: Weather Inquiries
    If you are given a place name or asked about weather, use the current_weather tool.
    - Execute the tool to get the current conditions.
    - Shortly describe the weather in a human readable form based on given data from the current_weather tool.

    ### Final Execution Rule
    If the user asks for both math and weather, you must run BOTH tools and include BOTH answers in your final response. Do not stop after the first result.
"""

result = await Runner.run(agent, "2^0.25. What's the weather today in Katowice?")
print(result.final_output)

Result: 1.189207

Current weather in Katowice: 
It's overcast with 11.3°C temperature, 11 mph winds coming from WSW and 100% cloud coverage.


13. Zabezpiecz agenta tak, aby nie odpowiadał na pytania, które nie dotyczą pogody lub potęgowania. Zademonstruj jego działanie.

In [52]:
agent.instructions = """
    You are a multi-tasking assistant. Your goal is to identify ALL distinct requests in the user's input and execute them.

    ### Task 1: Mathematical Calculations
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    If given nested expressions use power tool on them accordingly.
    Example process for 3^(8^(1/3)):
    Step 1: Calculate 8^(1/3). Call power(8, 0.333333). System returns 2.
    Step 2: Calculate 3^2 (using the result from Step 1). Call power(3, 2). System returns 9.
    **Math Output**: Return the answer in the format: Result: <result>. Result has to have 6 decimal places.

    ### Task 2: Weather Inquiries
    If you are given a place name or asked about weather, use the current_weather tool.
    - Execute the tool to get the current conditions.
    - Shortly describe the weather in a human readable form based on given data from the current_weather tool.

    ### Final Execution Rule
    If the user asks for both math and weather, you must run BOTH tools and include BOTH answers in your final response. Do not stop after the first result.
"""

result = await Runner.run(agent, "How to make a Lassagne?")
print(result.final_output)

print("---------------------")
print("new instruction")
print("------------------")
sleep(2)

agent.instructions = """
    You are a multi-tasking assistant. Your goal is to identify ALL distinct requests in the user's input and execute them.

    ### Task 1: Mathematical Calculations
    Use power tool on given numbers, if the given numbers are in formats like 1,23 or 1.23e-1 parse them to python correct format.
    For example 1,24^1,3 -> 1.24^1.3, 3.23e-2^1.25e-1 -> 0.0323^0.125.
    If the second given number is incorrect say that given numbers are incorrect.
    For example 4^(1/0) -> Given numbers are incorrect.
    If the user provides a fraction (e.g. 1/3), you MUST calculate its decimal value (e.g. 0.3333) BEFORE calling the tool.
    If given nested expressions use power tool on them accordingly.
    Example process for 3^(8^(1/3)):
    Step 1: Calculate 8^(1/3). Call power(8, 0.333333). System returns 2.
    Step 2: Calculate 3^2 (using the result from Step 1). Call power(3, 2). System returns 9.
    **Math Output**: Return the answer in the format: Result: <result>. Result has to have 6 decimal places.

    ### Task 2: Weather Inquiries
    If you are given a place name or asked about weather, use the current_weather tool.
    - Execute the tool to get the current conditions.
    - Shortly describe the weather in a human readable form based on given data from the current_weather tool.

    ### Final Execution Rule
    If the user asks for both math and weather, you must run BOTH tools and include BOTH answers in your final response. Do not stop after the first result.
    If user didn't ask about either math or weather then say that you can only do those two tasks. Do not stop after the first result.
"""

result = await Runner.run(agent, "How to make a Lassagne?")
print(result.final_output)

Since you haven't specified any executable tasks, I don't have any information to process.
---------------------
new instruction
------------------
I can only do math and weather tasks. If you provide me with either a math or weather related task, I can help you with that. Would you like to try a math problem or check the weather?


14. Zabezpiecz agenta tak, aby nie podawał pogody dla miejscowości leżących poza Polską. Zademonstruj jego działanie.


In [59]:
agent.instructions = """
    You are an intelligent assistant with two EXCLUSIVE modes.
    
    STEP 1: CLASSIFY THE USER INPUT
    - If the input contains numbers or math symbols -> Activate MODE MATH.
    - If the input is a city name or weather question -> Activate MODE WEATHER.
    
    STEP 2: EXECUTE THE SELECTED MODE (Do NOT run both unless explicitly asked)

    === MODE MATH ===
    1. Use the 'power' tool on the provided numbers.
    2. Format: "Result: <calculated_number>" (6 decimal places).
    3. NEVER output a random number if the input has no numbers.

    === MODE WEATHER ===
    1. Check if the city is in Poland.
    2. If NO (e.g., London, Berlin) -> Reply: "I can only check weather for cities in Poland." and STOP.
    3. If YES (e.g., Szczecin, Warsaw) -> Use 'current_weather' tool.
    4. Format: Describe the weather naturally (e.g., "The weather in Szczecin is...").
    5. CRITICAL: Do NOT print "Result: <number>" in this mode.

    ANTI-HALLUCINATION RULE:
    If the user asks for "Szczecin", do NOT call the power tool. Only call current_weather.
"""

print("--- Test 1: London ---")
result = await Runner.run(agent, "London")
print(result.final_output)

sleep(2)

print("\n--- Test 2: Szczecin ---")
result = await Runner.run(agent, "Szczecin")
print(result.final_output)

sleep(2)

print("\n--- Test 3: Math (Control) ---")
result = await Runner.run(agent, "2^3")
print(result.final_output)

--- Test 1: London ---
I can only check weather for cities in Poland.</function>

--- Test 2: Szczecin ---
The weather in Szczecin is light rain.

--- Test 3: Math (Control) ---
Result: 8.0


Agent RAG

1. Pobierz za pomocą biblioteki wikipedia treść strony Wikipedii o Układzie Słonecznym: https://en.wikipedia.org/wiki/Solar_System 

In [61]:
import wikipedia

page = wikipedia.WikipediaPage("Solar System").content

2. Podziel zawartość strony na 500-znakowe fragmenty (chunks) i umieść je w bazie wektorowej Chroma. Wykorzystaj domyślną funkcję osadzania (embedding) w Chroma.


In [63]:
import chromadb
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="solar_system")
chunks = [page[i:i + 500] for i in range(0, len(page), 500)]
print(len(chunks))

for id,chunk  in enumerate(chunks):
    collection.add(
        documents=chunk,
        ids=str(id),
    )

121


C:\Users\mateu\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:04<00:00, 19.4MiB/s]


3. Stwórz funkcję (narzędzie), które odpytuje bazę wektorową i zwraca top3 dokumenty najbardziej podobne do zapytania użytkownika.

In [64]:
from agents import agent, function_tool
@function_tool
def query(prompt):
    results = collection.query(query_texts=[prompt], n_results=3)
    return results

4. Stwórz agenta, który dla pytań związanych z Układem Słonecznych będzie korzystał z przygotowanego narzędzia. Zademonstruj działanie agenta.


In [ ]:
from agents import Agent, Runner, function_tool, set_default_openai_api, set_default_openai_client, set_tracing_disabled
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)
set_default_openai_client(client=client, use_for_tracing=False)
set_default_openai_api("chat_completions")
set_tracing_disabled(disabled=True)
agent = Agent(
    name="Assistant",
    instructions="""
    You're given query tool to search in database information about solar system.
    Based on the top 3 results from database answer user's questions.
    """,
    model="llama-3.1-8b-instant",
    tools=[query]
)

result = await Runner.run(agent, "What's the biggest planet in solar system?")
print(result.final_output)
result = await Runner.run(agent, "Does the planet Melmac exist?")
print(result.final_output)

The biggest planet in the solar system is Jupiter. The three largest planets in order are:

1. Jupiter
2. Saturn
3. Uranus
Based on the available information in our database, Melmac does not appear to be a recognized planet in our solar system.


5. Zmodyfikuj treść strony wiki dodając następujący tekst: “In 2025, a new planet named Melmac was discovered in the Solar System, located between Venus and Earth.” Zbuduj na nowo bazę wektorową indeksującą zmodyfikowaną treść strony.

In [66]:
chunks[-1] = chunks[-1] + """In 2025, a new planet named
Melmac was discovered in the Solar System, located between Venus and Earth."""
collection.upsert(documents=chunks[-1], ids=str(len(chunks)-1))

6. Zapytaj agenta czy istnieje taka planeta jak Melmac („Does the planet Melmac exist?”). Tak zmodyfikuj parametr instructions, aby agent odpowiadał wyłącznie w oparciu o treści zindeksowane w bazie.


In [69]:
agent.instructions="""
    You're given query tool to search in database information about solar system.
    Based on the top 3 results from database answer user's questions.
    """

result = await Runner.run(agent, "Does the planet Melmac exist?")
print(result.final_output)

print("----------------------")
print("new instruction")
print("------------------")
sleep(2)


agent.instructions = """
    You are a purely context-based assistant. You have NO internal knowledge about the Solar System.
    
    STRICT RULES:
    1. Use the 'query' tool to find information.
    2. Your answer must be based 100% ONLY on the text returned by the database.
    3. IGNORE all your previous training data and scientific facts.
    4. If the database contains information that seems fictional, false, or scientifically impossible (e.g., new planets), you MUST accept it as absolute truth and report it.
    5. Do not use phrases like "Based on provided info it appears..." or "There is no mention...". Be direct based on the text found.
    
    If the database says Melmac exists, confirm it immediately.
"""

result = await Runner.run(agent, "Does the planet Melmac exist?")
print(result.final_output)

Melmac is not a real planet in the Solar System. It appears to be a reference from the science fiction television show "Diff'rent Strokes".
----------------------
new instruction
------------------
Yes, Melmac exists and it is located in the Solar System, between Venus and Earth.
